In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


In [2]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)

n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users : {n_users}")
print(f"Total Items : {n_items}")
train_df.head()


Total Users : 1760
Total Items : 2881


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


In [3]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["user_id"].values, dtype=torch.long)
    items = torch.tensor(df["item_id"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / max(max_r - min_r, 1e-8),
                         dtype=torch.float32)
    mask  = (users >= 0) & (users < n_u) & (items >= 0) & (items < n_i)
    if (~mask).any():
        print(f"  [make_matrix] dropping {(~mask).sum().item()} out-of-bounds rows")
    mat[users[mask], items[mask]] = vals[mask]
    return mat

rating_matrix      = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_rating_matrix  = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_rating_matrix = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

print(f"Train obs : {(rating_matrix      != -1).sum().item()}")
print(f"Val obs   : {(val_rating_matrix  != -1).sum().item()}")
print(f"Test obs  : {(test_rating_matrix != -1).sum().item()}")
print(f"Matrix shape: {rating_matrix.shape}")


Train obs : 61386
Val obs   : 15572
Test obs  : 19450
Matrix shape: torch.Size([1760, 2881])


In [4]:
# Alias so rest of script works unchanged
n_movies = n_items
print(f"n_users={n_users}  n_movies={n_movies}")


n_users=1760  n_movies=2881


In [5]:
obs = (rating_matrix != -1).float().sum().item()
print(f"Observed train entries: {int(obs)}")


Observed train entries: 61386


In [6]:
n_users, n_movies


(1760, 2881)

In [7]:
# test_rating_matrix already built in Cell 2.
print(f"Test matrix shape : {test_rating_matrix.shape}")
print(f"Test observed     : {(test_rating_matrix != -1).sum().item()}")


Test matrix shape : torch.Size([1760, 2881])
Test observed     : 19450


In [8]:
class Loss(torch.nn.Module):
    def __init__(self, lam_u=0.3, lam_v=0.3):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
    
    def forward(self, matrix, u_features, v_features):
        non_zero_mask = (matrix != -1).type(torch.FloatTensor)
        predicted = torch.sigmoid(torch.mm(u_features, v_features.t()))
        
        diff = (matrix - predicted)**2
        prediction_error = torch.sum(diff*non_zero_mask)

        u_regularization = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_regularization = self.lam_v * torch.sum(v_features.norm(dim=1))

        return prediction_error + u_regularization + v_regularization, prediction_error

In [9]:
## Parameter Tuning — Grid Search
# Grid over lr, lam_u, latent_vectors.
# Uses val_rating_matrix for early stopping.

LR_GRID     = [0.001, 0.01,  0.1 ]
LAM_GRID    = [0.01,  0.1,   0.3 ]
LATENT_GRID = [10,    20,    40  ]
TUNE_EPOCHS = 30
TUNE_PAT    = 5

# RFRecF-specific constants (fixed during tuning)
threshold = 0.8   # prob of local update vs. server aggregation
eta       = 1.0   # aggregation step size

TUNE_CLIENTS = min(n_users, 200)
TUNE_M       = n_users // TUNE_CLIENTS

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)

grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)

    _uf = [torch.randn(TUNE_M, _K, requires_grad=True) for _ in range(TUNE_CLIENTS)]
    _mf = [torch.randn(n_movies, _K, requires_grad=True) for _ in range(TUNE_CLIENTS)]
    with torch.no_grad():
        for i in range(TUNE_CLIENTS):
            _uf[i].data.mul_(0.01)
            _mf[i].data.mul_(0.01)

    _loss_fn = Loss(lam_u=_lam, lam_v=0)
    _opts    = [torch.optim.Adam([_uf[i], _mf[i]], lr=_lr)
                for i in range(TUNE_CLIENTS)]
    _aver_mf = torch.zeros(n_movies, _K)

    _best_val, _wait = float('inf'), 0

    for _ep in range(TUNE_EPOCHS):
        p = torch.rand(1)
        if p > threshold:
            for i in range(TUNE_CLIENTS):
                _opts[i].zero_grad()
                _loss, _ = _loss_fn(
                    rating_matrix[:TUNE_CLIENTS * TUNE_M][i*TUNE_M:(i+1)*TUNE_M],
                    _uf[i], _mf[i])
                _loss.backward()
                _opts[i].step()
        else:
            _tmp = torch.zeros(n_movies, _K)
            with torch.no_grad():
                for i in range(TUNE_CLIENTS):
                    _tmp += _mf[i]
                _aver_mf = _tmp / TUNE_CLIENTS
                for i in range(TUNE_CLIENTS):
                    _mf[i] += eta * (_aver_mf - _mf[i])

        # Val RMSE using averaged movie features
        with torch.no_grad():
            _vm   = (val_rating_matrix[:TUNE_CLIENTS * TUNE_M] != -1).float()
            _n_v  = _vm.sum()
            if _n_v == 0:
                _val_rmse = float('inf')
            else:
                _se = 0.0
                for i in range(TUNE_CLIENTS):
                    _pred    = torch.sigmoid(torch.mm(_uf[i], _aver_mf.t()))
                    _pred_sc = (_pred * (max_rating - min_rating) + min_rating) * _vm[i*TUNE_M:(i+1)*TUNE_M]
                    _true_sc = val_rating_matrix[i*TUNE_M:(i+1)*TUNE_M] * _vm[i*TUNE_M:(i+1)*TUNE_M]
                    _se     += ((_pred_sc - _true_sc) ** 2).sum().item()
                _val_rmse = (_se / _n_v.item()) ** 0.5

        if _val_rmse < _best_val:
            _best_val, _wait = _val_rmse, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT:
                break

    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_val:>14.4f}{_marker}")

_best = min(grid_results, key=lambda x: x[0])
best_val_rmse_tune, best_lr, best_lam, best_K = _best
print(f"\nBest val RMSE  : {best_val_rmse_tune:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell.")


Grid search: 27 combinations x up to 30 epochs
  #       lr     lam     K   best val RMSE
------------------------------------------
  1   0.0010   0.010    10         12.4800 <--
  2   0.0010   0.010    20         12.4800 <--
  3   0.0010   0.010    40         12.4800 <--
  4   0.0010   0.100    10         12.4800
  5   0.0010   0.100    20         12.4800
  6   0.0010   0.100    40         12.4800
  7   0.0010   0.300    10         12.4800
  8   0.0010   0.300    20         12.4800
  9   0.0010   0.300    40         12.4800
 10   0.0100   0.010    10         12.4800
 11   0.0100   0.010    20         12.4800 <--
 12   0.0100   0.010    40         12.4752 <--
 13   0.0100   0.100    10         12.4800
 14   0.0100   0.100    20         12.4800
 15   0.0100   0.100    40         12.4800
 16   0.0100   0.300    10         12.4800
 17   0.0100   0.300    20         12.4800
 18   0.0100   0.300    40         12.4800
 19   0.1000   0.010    10         12.4795
 20   0.1000   0.010    20    

In [10]:
# ── Use tuned hyperparameters (fall back to defaults if grid search skipped) ─
try: latent_vectors
except NameError: latent_vectors = 20
try: lr
except NameError: lr = 0.01
try: lam
except NameError: lam = 0.1

# ── RFRecF training ───────────────────────────────────────────────────────────
alpha     = 0.025
threshold = 0.8
eta       = 1.0

num_client = min(n_users, 200)
m          = n_users // num_client
num_epoch  = 200
patience   = 10

rating_matrix_rfrecf = rating_matrix[:num_client * m]

torch.manual_seed(0)
user_features  = [torch.randn(m, latent_vectors, requires_grad=True)
                  for _ in range(num_client)]
movie_features = [torch.randn(n_movies, latent_vectors, requires_grad=True)
                  for _ in range(num_client)]
with torch.no_grad():
    for i in range(num_client):
        user_features[i].data.mul_(0.01)
        movie_features[i].data.mul_(0.01)

RFRecF_error       = Loss(lam_u=lam, lam_v=0)
optimizer_client_set = [torch.optim.Adam([user_features[i], movie_features[i]], lr=lr)
                        for i in range(num_client)]

aver_movie_features = torch.zeros(n_movies, latent_vectors)
error_list          = []
previous            = torch.zeros(n_movies, latent_vectors)

val_mask = (val_rating_matrix[:num_client * m] != -1).float()
n_val    = val_mask.sum()

best_val_rmse    = float('inf')
patience_count   = 0
best_aver_mf     = None
best_user_states = None

for step in range(num_epoch):
    aver_loss             = 0
    aver_prediction_error = 0

    p = torch.rand(1)
    if p > threshold:
        # local update
        for i in range(num_client):
            optimizer_client_set[i].zero_grad()
            loss, prediction_error = RFRecF_error(
                rating_matrix_rfrecf[i*m:(i+1)*m], user_features[i], movie_features[i])
            aver_loss             += loss / num_client
            aver_prediction_error += prediction_error / num_client
            loss.backward()
            optimizer_client_set[i].step()
    else:
        # server aggregation: average movie features
        tmp = torch.zeros(n_movies, latent_vectors)
        with torch.no_grad():
            for i in range(num_client):
                tmp += movie_features[i]
            aver_movie_features = tmp / num_client
            for i in range(num_client):
                movie_features[i] += eta * (aver_movie_features - movie_features[i])

    # convergence tracker
    error = torch.norm(aver_movie_features - previous, p=2) /             torch.norm(previous, p=2).clamp(min=1e-8)
    error_list.append(error.detach().numpy())
    previous = aver_movie_features.clone()

    # val RMSE for early stopping
    with torch.no_grad():
        se = 0.0
        for i in range(num_client):
            pred    = torch.sigmoid(torch.mm(user_features[i], aver_movie_features.t()))
            pred_sc = (pred * (max_rating - min_rating) + min_rating) * val_mask[i*m:(i+1)*m]
            true_sc = val_rating_matrix[i*m:(i+1)*m] * val_mask[i*m:(i+1)*m]
            se     += ((pred_sc - true_sc) ** 2).sum().item()
        val_rmse = (se / n_val.item()) ** 0.5 if n_val > 0 else float('inf')

    if step % 10 == 0:
        print(f"Step {step:3d} | avg pred error: {aver_prediction_error:.4f} "
              f"| val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse    = val_rmse
        best_aver_mf     = aver_movie_features.clone()
        best_user_states = [uf.data.clone() for uf in user_features]
        patience_count   = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at step {step}.")
            break

# Restore best checkpoint
aver_movie_features = best_aver_mf
for uf, state in zip(user_features, best_user_states):
    uf.data.copy_(state)

print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Step   0 | avg pred error: 0.0000 | val RMSE: 12.4800
Step  10 | avg pred error: 64.7376 | val RMSE: 12.4655
Step  20 | avg pred error: 0.0000 | val RMSE: 11.7665
Step  30 | avg pred error: 0.0000 | val RMSE: 11.7665
Early stopping at step 30.

Best val RMSE: 11.7665


In [11]:
# test_rating_matrix already built in Cell 2.
print(f"Test matrix shape : {test_rating_matrix.shape}")
print(f"Test observed     : {(test_rating_matrix != -1).sum().item()}")


Test matrix shape : torch.Size([1760, 2881])
Test observed     : 19450


In [12]:
n_users, n_movies

(1760, 2881)

In [13]:
non_zero_mask = (test_rating_matrix[:num_client * m] != -1).type(torch.FloatTensor)
num           = torch.sum(non_zero_mask)

# Use averaged movie features from training
movie_features_eval = aver_movie_features

def Error(matrix, u_features, v_features):
    predicted_ratings = torch.sigmoid(torch.mm(u_features, v_features.t()))
    pred    = (predicted_ratings * (max_rating - min_rating) + min_rating)               * non_zero_mask[i*m:(i+1)*m]
    actual  = matrix * non_zero_mask[i*m:(i+1)*m]
    AE_diff = torch.abs(pred - actual)
    SE_diff = (pred - actual) ** 2
    n_nz    = torch.sum(non_zero_mask[i*m:(i+1)*m])
    return torch.sum(AE_diff), torch.sum(SE_diff), n_nz

AE_error  = 0
SE_error  = 0
num_non_zero = 0

for i in range(num_client):
    ae, se, nz = Error(test_rating_matrix[i*m:(i+1)*m],
                       user_features[i], movie_features_eval)
    AE_error     += ae
    SE_error     += se
    num_non_zero += nz

test_MAE  = AE_error  / num_non_zero
test_RMSE = torch.sqrt(SE_error / num_non_zero)
print(f"Test MAE  : {test_MAE.item():.4f}")
print(f"Test RMSE : {test_RMSE.item():.4f}")


Test MAE  : 11.7283
Test RMSE : 11.7600


In [14]:
# Evaluation is performed in Cell 12 above.


In [15]:
# Quick debug: inspect one client's Error output
i = 1
Error(test_rating_matrix[i*m:(i+1)*m], user_features[i], movie_features_eval)


(tensor(1265.6929, grad_fn=<SumBackward0>),
 tensor(14688.1602, grad_fn=<SumBackward0>),
 tensor(110.))

In [16]:
aver_movie_features


tensor([[ 0.0431,  0.0154,  0.0930,  ...,  0.0482, -0.0966,  0.0682],
        [ 0.0276,  0.0277,  0.0834,  ...,  0.0184, -0.0525,  0.0749],
        [ 0.0094, -0.0007,  0.0181,  ...,  0.0055, -0.0091, -0.0012],
        ...,
        [ 0.0037,  0.0020,  0.0069,  ..., -0.0014, -0.0066, -0.0089],
        [ 0.0042,  0.0033,  0.0082,  ...,  0.0076, -0.0075, -0.0017],
        [-0.0037,  0.0041,  0.0043,  ..., -0.0014, -0.0001,  0.0033]])